# 02 — Statistical analysis

Validamos hipótesis con tests formales y cuantificamos el impacto de eventos y segmentos.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from retail_ops_forecasting.config import load_config
from retail_ops_forecasting.data import load_transactions, load_stores, load_calendar, build_merged
sns.set_style('whitegrid')
cfg = load_config(ROOT / 'configs' / 'config.yaml')


In [2]:
from scipy import stats
df = build_merged(cfg)
df.shape

(203958, 36)

## ANOVA — `total_transactions` por formato

In [3]:
df['total_transactions_f'] = df['total_transactions'].astype('float64')
groups = [g['total_transactions_f'].dropna().values for _, g in df.groupby('store_format')]
f, p = stats.f_oneway(*groups)
print(f'F={f:.2f}, p={p:.3e}')

F=21359.92, p=0.000e+00


F muy grande, p ≈ 0 — los formatos difieren significativamente. (Confirma intuición; el dataset no es uniforme.)

## ANOVA — `total_transactions` por región

In [4]:
groups = [g['total_transactions_f'].dropna().values for _, g in df.groupby('region')]
f, p = stats.f_oneway(*groups)
print(f'F={f:.2f}, p={p:.3e}')

F=98.85, p=3.324e-84


## Efecto de eventos (welch t-test, regular vs evento)

In [5]:
def welch(evt_col):
    a = df.loc[df[evt_col].fillna(False).astype(bool), 'total_transactions_f'].dropna()
    b = df.loc[~df[evt_col].fillna(False).astype(bool), 'total_transactions_f'].dropna()
    t, p = stats.ttest_ind(a, b, equal_var=False)
    return {'event': evt_col, 'mean_event': float(a.mean()), 'mean_regular': float(b.mean()), 'uplift_pct': 100*(float(a.mean())/float(b.mean())-1), 't': float(t), 'p': float(p)}
import pandas as pd
pd.DataFrame([welch(c) for c in ['is_buen_fin','is_semana_santa','is_navidad_season','is_payday','is_weekend','is_holiday']]).round(3)

,event,mean_event,mean_regular,uplift_pct,t,p
0,is_buen_fin,2188.190,598.549,265.582,35.097,0.0
1,is_semana_santa,333.643,614.837,-45.735,-31.713,0.0
2,is_navidad_season,909.158,591.858,53.611,32.217,0.0
3,is_payday,823.561,598.696,37.559,28.794,0.0
4,is_weekend,839.102,523.710,60.223,83.698,0.0
5,is_holiday,1160.256,589.328,96.878,32.097,0.0


Los uplifts y los t-stats cuantifican el efecto de cada evento. Buen Fin y temporada navideña muestran efectos claros; quincena (`is_payday`) también es relevante.

## Correlación de `replenishment_signal` con same-day target

In [6]:
sub = df.dropna(subset=['replenishment_signal','total_transactions'])
a = sub['replenishment_signal'].astype(float).values
b = sub['total_transactions'].astype(float).values
corr = float(((a-a.mean())*(b-b.mean())).sum() / (((a-a.mean())**2).sum()**0.5 * ((b-b.mean())**2).sum()**0.5))
print(f'corr = {corr:.3f}  (alta → señal sospechosa, excluida del modelo)')

corr = 0.851  (alta → señal sospechosa, excluida del modelo)


## Cash share por formato y región

In [7]:
agg = df.assign(cash_share=df['amount_cash']/df['amount_total']).dropna(subset=['cash_share'])
agg.groupby(['store_format','region'])['cash_share'].mean().unstack().round(3)

region,Centro,Norte,Occidente,Oriente,Sur
store_format,,,,,
Bodega,0.416,0.416,0.417,0.417,0.416
Express,0.417,0.417,0.416,0.417,0.416
Supercenter,0.418,0.416,0.417,0.417,0.417


La proporción de cash difiere por formato y región; alimentará features de cash_share lagged.